# Find the Language I Am Speaking: spoken language ID for 12 Nigerian languages

**Team Ohùn** (Codabench: datapsalm): Oludare Samuel, Victoria Fagbemiro, Sumayah Adegbite · DSN 2026 AI Bootcamp hackathon, ML/Agentic AI track

Given a 2–30 second voice clip, predict which of 12 Nigerian languages is spoken (Yoruba, Igbo, Hausa, Eggon, Annang, Efik, Ibibio, Idoma, Igala, Nupe, Tiv, Urhobo) or `other`.
The score is `language-weighted F1 × (0.5 + 0.5 × efficiency)`, where efficiency rewards a small, light, fast model.

**Our approach, in short**
1. Fine-tune **AfriHuBERT** (94.6M parameters), a speech model pre-trained on African languages.
2. Train robustly: noise, phone-line and echo effects, speed changes, balanced sampling, and, for the final model, all labelled data.
3. Add **openly licensed recordings from new speakers** of our weakest languages: 19 Ibibio speakers (Mozilla Common Voice, CC0) and 2 Efik speakers (CC-BY-4.0). Neither comes from the corpora the test set was drawn from.
4. Average the weights of two fine-tuned models (**model soup**) at no cost to size or speed.

This notebook walks through each part with the project's own code (`common.py`, `train.py`, `soup.py`), then runs the final model.
Full code and every run's exact command: https://github.com/Dhapor/dsn2026-nigerian-language-id

In [1]:
import json, os, time
os.environ["TQDM_DISABLE"] = "1"
from collections import Counter
import numpy as np
import torch
from common import LABELS, L2I, SR, Split, weighted_f1, efficiency, final_score, augment, CLASS_W
import transformers
transformers.logging.set_verbosity_error(); transformers.utils.logging.disable_progress_bar()
print("torch", torch.__version__, "| classes:", LABELS)

torch 2.11.0 | classes: ['yor', 'ibo', 'hau', 'ego', 'anw', 'efi', 'ibb', 'idu', 'igl', 'nup', 'tiv', 'urh', 'other']


## 1. The data, and why it is hard

The official data has 24,774 training and 4,352 validation clips. `prep.py` packs all audio into one int16 file per split so training can read any slice instantly. Below: hours and speakers per language.

In [2]:
tr, va = Split("train"), Split("validation")
print(f"{'class':6} {'train h':>8} {'val h':>6} {'train clips':>12} {'speakers':>9}  weight")
for c, lab in enumerate(LABELS):
    m = tr.labels == c
    spk = {s for s, k in zip(tr.speakers, m) if k and s not in ("", "None")}
    print(f"{lab:6} {tr.lengths[m].sum()/SR/3600:8.2f} {va.lengths[va.labels==c].sum()/SR/3600:6.2f} "
          f"{m.sum():12d} {len(spk) if spk else 'n/a':>9}  {CLASS_W[c]:.3f}")
print("sources:", Counter(tr.sources))
print("n/a = the source has no speaker IDs")

class   train h  val h  train clips  speakers  weight
yor        9.05   0.88         3160       n/a  0.043
ibo        7.96   2.08         2592         7  0.043
hau        8.31   1.42         2755         3  0.043
ego        7.32   1.05         1631        10  0.087
anw        8.12   1.83         1912        20  0.087
efi        1.42   0.15          483         3  0.087
ibb        1.76   0.28          607         2  0.087
idu        7.16   2.12         1649        11  0.087
igl        0.48   0.12          170         2  0.087
nup        0.50   0.00          176         1  0.087
tiv        4.27   0.63         1474         2  0.087
urh        9.22   1.57         3122       n/a  0.087
other     12.98   1.85         5043        51  0.087
sources: Counter({'afrispeech_ipa': 15817, 'hypa_voices': 4722, 'omnilingual_asr': 4235})
n/a = the source has no speaker IDs


Two things stand out:
* **Few speakers.** Igala and Nupe have about half an hour each; Ibibio has 2 speakers, Efik 3.
* **One source per language.** Some languages come from a single recording source, so a model can learn the microphone or studio instead of the language. Early on, a model scored 0.85 F1 on Yoruba in validation but 0.38 on the test set for exactly this reason.

The score weights the nine smaller languages and `other` twice as much as Yoruba, Igbo and Hausa (column *weight*).

## 2. The official metric

We re-implemented the competition's metric so our validation numbers mean the same as the leaderboard's.

In [3]:
import inspect
print(inspect.getsource(weighted_f1))
print(inspect.getsource(efficiency))
# Our final model's measured efficiency on the organisers' hardware was 0.790 and its test F1 0.6247:
print("final score ≈", round(final_score(0.6247, 0.790), 3))

def weighted_f1(y_true, y_pred):
    """Language-weighted F1 exactly as described on the Evaluation page."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    f1s = np.zeros(len(LABELS))
    for c in range(len(LABELS)):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        f1s[c] = 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)
    return float(np.sum(f1s * CLASS_W)), f1s

def efficiency(params, repo_bytes, peak_bytes, sec_per_clip):
    size = ((1 - params / 350e6) + (1 - repo_bytes / 1.5e9)) / 2
    mem = 1 - peak_bytes / 4e9
    t = max(sec_per_clip, 0.01)
    speed = 1 - math.log(t / 0.01) / math.log(0.5 / 0.01)
    eff = (size + mem + speed) / 3
    return eff, dict(size=size, mem=mem, speed=speed)

final score ≈ 0.559


## 3. Augmentation: making a few speakers sound like many

The test set has clean, noisy and phone recordings. `augment()` randomly adds a phone-line channel, coloured noise or background chatter, echo and a gain change. Speed perturbation (in `TrainDS`) also shifts pitch and tempo by up to ±15%.

In [4]:
rng = np.random.default_rng(0)
i = int(np.flatnonzero(tr.labels == L2I["tiv"])[0])
x = tr.wav(i, 0, 4 * SR)
print(f"original: loudness (rms) {np.sqrt(np.mean(x**2)):.3f}")
for k in range(5):
    y = augment(x.copy(), rng)
    sim = np.corrcoef(x, y)[0, 1]
    print(f"augmented copy {k}: loudness {np.sqrt(np.mean(y**2)):.3f}, similarity to the original {sim:.2f}")
print("similarity 1.00 = only the volume changed; lower = noise, phone line or echo was added")

original: loudness (rms) 0.091
augmented copy 0: loudness 0.025, similarity to the original 0.98
augmented copy 1: loudness 0.082, similarity to the original 1.00


augmented copy 2: loudness 0.039, similarity to the original 0.40
augmented copy 3: loudness 0.118, similarity to the original 1.00
augmented copy 4: loudness 0.161, similarity to the original 0.99
similarity 1.00 = only the volume changed; lower = noise, phone line or echo was added


## 4. Balanced sampling

Each training clip is drawn with weight `count(label) ** -alpha`. With `alpha = 0.5`, a rare language is seen far more often than its raw share, without being repeated so often that the model memorises its few speakers.

In [5]:
counts = np.bincount(tr.labels, minlength=len(LABELS))
for alpha in (0.0, 0.5):
    w = counts.astype(float) ** -alpha * counts      # total weight per class
    p = w / w.sum()
    print(f"alpha={alpha}: " + "  ".join(f"{l} {100*q:4.1f}%" for l, q in zip(LABELS, p)))

alpha=0.0: yor 12.8%  ibo 10.5%  hau 11.1%  ego  6.6%  anw  7.7%  efi  1.9%  ibb  2.5%  idu  6.7%  igl  0.7%  nup  0.7%  tiv  5.9%  urh 12.6%  other 20.4%
alpha=0.5: yor 10.8%  ibo  9.7%  hau 10.0%  ego  7.7%  anw  8.4%  efi  4.2%  ibb  4.7%  idu  7.8%  igl  2.5%  nup  2.5%  tiv  7.3%  urh 10.7%  other 13.6%


## 5. Training setup

`train.py` loads AfriHuBERT with a 13-way classification head, trains only the head for 300 steps, then fine-tunes the whole model at a learning rate of 5e-5 with cross-entropy (label smoothing 0.1), random 2–8 s crops and batches of 16. Training runs on an Apple M4 Max GPU and takes about 40 minutes, so it is not re-run here. These are the exact settings of the final model (F07):

In [6]:
args = json.load(open("runs/F07/args.json"))
for k in ["base", "full", "ext", "bs", "lr", "head_lr", "epochs", "steps_per_epoch", "min_crop", "max_crop", "crop_step", "speed", "aug", "alpha", "smooth", "freeze_steps"]:
    print(f"{k:16} {args[k]}")

base             ajesujoba/AfriHuBERT
full             True
ext              ['ext_efik', 'ext_cv_ibb']
bs               16
lr               5e-05
head_lr          0.001
epochs           3.0
steps_per_epoch  1500
min_crop         2.0
max_crop         8.0
crop_step        1.0
speed            0.6
aug              0.8
alpha            0.5
smooth           0.1
freeze_steps     300


The core of one training step:

```python
logits = model(**inp).logits
loss = F.cross_entropy(logits.float(), y, label_smoothing=0.1)
loss.backward(); clip_grad_norm_(model.parameters(), 5.0); opt.step()
```

## 6. Choosing the recipe on validation

We compared two recipes on validation (models trained on the training split only): a *debiased* one that fights the source shortcut (source-balanced sampling, a source adversary and random channel EQ) and a *plain* one. Below we score their saved validation predictions with the official metric, then their weight average (*model soup*).

In [7]:
def report(name, path):
    d = np.load(path)
    f1, per = weighted_f1(d["y"], d["logits"].argmax(1))
    print(f"{name:22} weighted F1 {f1:.3f} | " + " ".join(f"{LABELS[c]} {per[c]:.2f}" for c in range(len(LABELS))))
    return d

for name, path in [("A01 debiased", "runs/soup_eval/a01.npz"), ("A03 plain", "runs/soup_eval/a03.npz"), ("soup(A01, A03)", "runs/soup_eval/soup_a01_a03.npz")]:
    report(name, path)
print("(validation split only; it has no Nupe clips, so nup shows 0 for every model)")

A01 debiased           weighted F1 0.594 | yor 0.88 ibo 0.99 hau 0.85 ego 0.92 anw 0.87 efi 0.11 ibb 0.00 idu 0.93 igl 0.43 nup 0.00 tiv 0.35 urh 0.97 other 0.88
A03 plain              weighted F1 0.607 | yor 0.88 ibo 0.98 hau 0.96 ego 0.96 anw 0.86 efi 0.12 ibb 0.00 idu 0.93 igl 0.35 nup 0.00 tiv 0.46 urh 0.99 other 0.92
soup(A01, A03)         weighted F1 0.626 | yor 0.90 ibo 0.98 hau 0.88 ego 0.98 anw 0.91 efi 0.18 ibb 0.03 idu 0.93 igl 0.50 nup 0.00 tiv 0.42 urh 0.98 other 0.90
(validation split only; it has no Nupe clips, so nup shows 0 for every model)


The soup beats both recipes, and gains most on Annang, Igala and Efik. It is built by averaging every weight of the two models (`soup.py`):

In [8]:
import soup
print(inspect.getsource(soup.average))

def average(paths):
    ms = [AutoModelForAudioClassification.from_pretrained(p, dtype=torch.float32) for p in paths]
    sds = [m.state_dict() for m in ms]
    assert all(sd.keys() == sds[0].keys() for sd in sds), "checkpoints must share one architecture"
    ms[0].load_state_dict({k: sum(sd[k].float() for sd in sds) / len(sds) for k in sds[0]})
    return ms[0]



## 7. Results on the hidden test set

Every row is a real Codabench submission.

| Model | Change | Test F1 | Efficiency | Final |
|---|---|---|---|---|
| Whisper-base encoder | Baseline | 0.523 | 0.886 | 0.493 |
| Whisper-medium encoder | Bigger model | 0.613 | 0.667 | 0.511 |
| AfriHuBERT, train only | African base, debiased recipe | 0.529 | 0.790 | 0.474 |
| AfriHuBERT, train + validation | All labelled speakers | 0.593 | 0.790 | 0.530 |
| **AfriHuBERT + external data (F07)** | **Plain recipe, +21 new speakers** | **0.625** | **0.790** | **0.559** |
| Soup of F05 and F07 | Weight average | pending | | |

## 8. Using the final model

The model is a standard Hugging Face audio classifier, stored in fp16 (0.19 GB). It loads with the `transformers` Auto classes, no custom code. Here it predicts a few validation clips (note: F07 was trained on train + validation, so this only demonstrates usage, not accuracy).

In [9]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
repo, rev = "datapsalm/dsn2026-lid-f07", "1ac06ea69be473693a29981b12cab0ec787dbba2"
model = AutoModelForAudioClassification.from_pretrained(repo, revision=rev, dtype=torch.float32).eval()
fe = AutoFeatureExtractor.from_pretrained(repo, revision=rev)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

rng = np.random.default_rng(1)
picks = [int(rng.choice(np.flatnonzero(va.labels == c))) for c in range(len(LABELS)) if (va.labels == c).any()]
times = []
with torch.no_grad():
    for i in picks:
        x = va.wav(i, 0, min(va.lengths[i], 12 * SR))
        t = time.time()
        probs = model(**fe([x], sampling_rate=SR, return_tensors="pt")).logits.softmax(-1)[0]
        times.append(time.time() - t)
        top = probs.argsort(descending=True)[:2]
        print(f"true {LABELS[va.labels[i]]:5} -> predicted {model.config.id2label[int(top[0])]:5} ({probs[top[0]]:.2f}), "
              f"runner-up {model.config.id2label[int(top[1])]} ({probs[top[1]]:.2f}), {len(x)/SR:4.1f} s clip")
print(f"median {1000*np.median(times):.0f} ms per clip on this laptop's CPU")

94.6M parameters
true yor   -> predicted yor   (0.90), runner-up other (0.01),  6.9 s clip


true ibo   -> predicted ibo   (0.90), runner-up tiv (0.01), 12.0 s clip
true hau   -> predicted hau   (0.93), runner-up ibb (0.01), 11.1 s clip
true ego   -> predicted ego   (0.86), runner-up tiv (0.03), 12.0 s clip


true anw   -> predicted anw   (0.90), runner-up tiv (0.02), 12.0 s clip
true efi   -> predicted efi   (0.86), runner-up tiv (0.03), 12.0 s clip
true ibb   -> predicted ibb   (0.84), runner-up igl (0.09),  6.0 s clip


true idu   -> predicted idu   (0.89), runner-up tiv (0.02), 12.0 s clip
true igl   -> predicted igl   (0.96), runner-up tiv (0.01), 12.0 s clip
true tiv   -> predicted tiv   (0.96), runner-up idu (0.01), 12.0 s clip


true urh   -> predicted urh   (0.89), runner-up tiv (0.02), 12.0 s clip
true other -> predicted other (0.91), runner-up anw (0.01), 11.1 s clip
median 92 ms per clip on this laptop's CPU


## 9. What we learned

* **Data is the cheap win.** Training on all labelled speakers added +0.06 test F1 at no efficiency cost. A 15× bigger Whisper gained 0.09 F1, but the efficiency penalty cancelled most of it (final 0.493 → 0.511).
* **Validation can mislead** when each language comes from one recording source; the test set exposed a source shortcut that validation hid.
* **Closely related languages collide.** Efik, Ibibio and Annang are related, and Annang (20 training speakers) absorbs the other two. The fix is more everyday recordings from many speakers.

**Next:** record more voices for Ibibio, Efik, Igala and Nupe; train the model to separate the three related languages; and use it as a fast *language router* that sends each voice clip to the right speech-recognition model.

**Credits.** Base model: [ajesujoba/AfriHuBERT](https://huggingface.co/ajesujoba/AfriHuBERT). Data: [Data-Science-Nigeria/dsn-2026-lid-train](https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train) (afrispeech_ipa CC-BY-4.0, hypa_voices Apache-2.0, omnilingual_asr CC-BY-4.0), [Mozilla Common Voice 27.0 Ibibio](https://mozilladatacollective.com/organization/cmfh0j9o10006ns07jq45h7xk) (CC0-1.0), [offiongbassey/efik_speech_dataset](https://huggingface.co/datasets/offiongbassey/efik_speech_dataset) (CC-BY-4.0).